# 개별종목 데이터 분석

## 목표

2023년 1월 1일 기준
코스피 종목 중 시가총액 5000억 이상인 종목에 해당하여,

아래의 데이터에 의해서 각 종목의

1. 신용잔고, 신용비율
2. 외국인, 기관 순매수,순매도 데이터
3. 공매도 잔고, 대차잔고
4. 거래대금 , 거래량

에 대해서 (1) 각 종목의 주가가 어떤 흐름을 나타내는지.

또한 (2) KOSPI 200 지수에 대하여, 지수는 어떤 흐름을 나타내는지.

## 입력파일

1. 2023-01-02 현재 코스피 종목 중 시가총액 5000억 이상 종목 (287개)
   - 시총5000억이상_20230102.csv
2. 외국인/기관 순매수 데이터
   - 기관_외국인_순매수_20230102_20251231.csv
3. 공매도 잔고
   - 공매도잔고_20230102_20251231.csv
4. 거래대금, 거래량
   - OHLCV_20230102_20251231.csv
5. 대차 잔고
   - 대차잔고_20230102_20251231.csv
6. 신용잔고, 대주잔고
   - 신용잔고_대주잔고_20230102_20251231.csv

# 1단계. 선행성/예측령 체크

**목표**  
- 팩터(인디케이터) 값이 미래 수익률(fwd_ret) 과 단면에서 얼마나 관계가 있는지(IC 평균/분산/히트율) 확인
- 결과: 살릴 팩터 후보만 다음 단계로 넘김

**실행 코드 (병합 → 팩터 생성 → 미래수익률 → IC 계산)**

In [2]:
import pandas as pd
import numpy as np

# ----------------------------
# 0) 로드
# ----------------------------
PATH_OHLCV = "OHLCV_20230102_20251231.csv"
PATH_FLOW  = "기관_외국인_순매수_20230102_20251231.csv"
PATH_SHORT = "공매도잔고_20230102_20251231.csv"
PATH_BORROW= "대차잔고_20230102_20251231.csv"
PATH_MARGIN= "신용잔고_대주잔고_20230102_20251231.csv"

ohlcv  = pd.read_csv(PATH_OHLCV,  parse_dates=["date"])
flow   = pd.read_csv(PATH_FLOW,   parse_dates=["date"])
short  = pd.read_csv(PATH_SHORT,  parse_dates=["date"])
borrow = pd.read_csv(PATH_BORROW, parse_dates=["date"])
margin = pd.read_csv(PATH_MARGIN, parse_dates=["date"])

# ----------------------------
# 1) 병합 (date, ticker)
# ----------------------------
df = ohlcv.merge(flow,   on=["date","ticker","name"], how="left") \
         .merge(short,  on=["date","ticker","name"], how="left") \
         .merge(borrow, on=["date","ticker","name"], how="left") \
         .merge(margin, on=["date","ticker","name"], how="left")

df = df.sort_values(["ticker","date"]).reset_index(drop=True)

# ----------------------------
# 2) 기본 수익률/미래수익률
# ----------------------------
df["ret_1"]  = df.groupby("ticker")["종가"].pct_change()
df["ret_5"]  = df.groupby("ticker")["종가"].pct_change(5)
df["ret_20"] = df.groupby("ticker")["종가"].pct_change(20)

# 미래수익률(예측 타깃)
df["fwd_ret_1"]  = df.groupby("ticker")["종가"].pct_change(-1)   # t -> t+1
df["fwd_ret_5"]  = df.groupby("ticker")["종가"].pct_change(-5)
df["fwd_ret_20"] = df.groupby("ticker")["종가"].pct_change(-20)

# ----------------------------
# 3) 인디케이터(팩터) 후보 생성 (최소 핵심 세트)
#    - 거래대금으로 정규화된 수급
#    - 잔고류는 변화량
# ----------------------------
# (1) Flow intensity
df["foreign_flow"] = df["외국인합계"] / df["거래대금"]
df["inst_flow"]    = df["기관합계"]   / df["거래대금"]
df["smart_flow"]   = (df["외국인합계"] + df["기관합계"]) / df["거래대금"]

# rolling (5, 20)
for c in ["foreign_flow","inst_flow","smart_flow"]:
    df[f"{c}_roll5"]  = df.groupby("ticker")[c].transform(lambda x: x.rolling(5).mean())
    df[f"{c}_roll20"] = df.groupby("ticker")[c].transform(lambda x: x.rolling(20).mean())

# (2) Short / Borrow / Margin 변화량
# 공매도 비중: 파일에 '비중'이 이미 있음
df["d_short_ratio_5"] = df.groupby("ticker")["비중"].diff(5)

# 대차잔고(주식수/금액): 증가 신호
df["d_borrow_sh_20"]  = df.groupby("ticker")["대차잔고주식수"].diff(20)
df["d_borrow_amt_20"] = df.groupby("ticker")["대차잔고금액"].diff(20)

# 신용비율: 과열/청산
df["d_margin_ratio_20"] = df.groupby("ticker")["신용비율"].diff(20)

# ----------------------------
# 4) 단면(IC) 계산 함수
# ----------------------------
def daily_ic(panel_df, factor_col, target_col):
    """
    날짜별 단면 Spearman 상관 -> IC 시계열 반환
    """
    def _ic_one_day(g):
        x = g[factor_col]
        y = g[target_col]
        ok = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)
        if ok.sum() < 30:  # 단면 표본 너무 적으면 스킵
            return np.nan
        return x[ok].rank().corr(y[ok].rank())  # Spearman = rank corr

    # 방법 A: include_groups=False (pandas 최신)
    try:
        return panel_df.groupby("date").apply(_ic_one_day, include_groups=False)
    except TypeError:
        # 방법 B: 필요한 컬럼만 선택해서 apply (구버전 호환)
        return panel_df[["date", factor_col, target_col]].groupby("date").apply(_ic_one_day)
    # return panel_df.groupby("date").apply(_ic_one_day)

def summarize_ic(ic_series):
    ic = ic_series.dropna()
    if len(ic) == 0:
        return {"n":0, "ic_mean":np.nan, "ic_std":np.nan, "ic_ir":np.nan, "hit_rate":np.nan}
    ic_mean = ic.mean()
    ic_std  = ic.std(ddof=1)
    ic_ir   = ic_mean / ic_std if ic_std > 0 else np.nan
    hit     = (ic > 0).mean()
    return {"n":len(ic), "ic_mean":ic_mean, "ic_std":ic_std, "ic_ir":ic_ir, "hit_rate":hit}

# ----------------------------
# 5) 팩터 리스트 대상으로 IC 요약
# ----------------------------
factors = [
    "foreign_flow_roll5","foreign_flow_roll20",
    "inst_flow_roll5","inst_flow_roll20",
    "smart_flow_roll20",
    "d_short_ratio_5",
    "d_borrow_sh_20","d_borrow_amt_20",
    "d_margin_ratio_20",
    "ret_5","ret_20"   # 가격 모멘텀 베이스라인도 같이 비교
]

target = "fwd_ret_20"   # 먼저 20영업일(약 1개월) 기준으로 보자

rows = []
for f in factors:
    ic_ts = daily_ic(df, f, target)
    s = summarize_ic(ic_ts)
    rows.append([f, s["n"], s["ic_mean"], s["ic_std"], s["ic_ir"], s["hit_rate"]])

ic_summary = pd.DataFrame(rows, columns=["factor","n_days","ic_mean","ic_std","ic_ir","hit_rate"]) \
              .sort_values("ic_ir", ascending=False)

print(ic_summary.head(20))


                 factor  n_days   ic_mean    ic_std     ic_ir  hit_rate
8     d_margin_ratio_20     691  0.012172  0.079497  0.153113  0.561505
5       d_short_ratio_5     706  0.008891  0.070764  0.125647  0.572238
2       inst_flow_roll5     707  0.010331  0.090958  0.113579  0.543140
3      inst_flow_roll20     692  0.010022  0.092404  0.108454  0.502890
10               ret_20     691  0.007293  0.150904  0.048332  0.520984
9                 ret_5     706  0.003605  0.131916  0.027325  0.511331
4     smart_flow_roll20     692 -0.003053  0.115533 -0.026426  0.440751
7       d_borrow_amt_20     687 -0.006040  0.101779 -0.059349  0.486172
0    foreign_flow_roll5     707 -0.008294  0.075917 -0.109244  0.462518
1   foreign_flow_roll20     692 -0.012950  0.082521 -0.156927  0.475434
6        d_borrow_sh_20     687 -0.011473  0.068745 -0.166890  0.438137


### 1단계 결과 해석

타깃이 fwd_ret_20(약 1개월)일 때:

**“롱 성격(+)” 후보**

- d_margin_ratio_20: IC_mean +0.0122, IR 0.153, hit 0.562
- d_short_ratio_5: IC_mean +0.0089, IR 0.126, hit 0.572
- inst_flow_roll5: IC_mean +0.0103, IR 0.114, hit 0.543
- inst_flow_roll20: IC_mean +0.0100, IR 0.108, hit 0.503

→ “신용비율 변화(20일)”, “공매도비중 변화(5일)”, “기관 순매수 강도”가  
  미래 20일 수익률과 약하게나마 양(+)의 관계.

**“숏/회피 성격(-)” 후보**

- foreign_flow_roll20: IC_mean -0.0130, IR -0.157
- d_borrow_sh_20: IC_mean -0.0115, IR -0.167

→ 외국인 수급, 대차 변화는 이 구간에선 반대로 작동하는 경향이 보임  
  (특히 외국인은 -가 꽤 일관).

> 결론: 2단계는 우선 상위 6개(절대 IR 기준) 정도로 조합을 만들고,  
>      방향(+) / (-) 섞어 하나의 score로 합치면 됨.

---

## 2단계: 다중공선성 제거 + 팩터 조합

**목표**  
- 1단계 통과한 팩터들끼리 겹치는 정보(상관)를 줄이고
- 조합 신호를 하나 만든다 (점수/확률)

**실행코드**  
- (A) 공선정 제거 (스피어만 상관 기반)
- (B) Ridge로 조합 스코어 생성
- (C) (선택)시계열 누수 방지: 워크포워드 학습으로 스코어 생성

In [4]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


# =========================================================
# 4.2 STEP: 공선성 제거 + 팩터 조합(score) 생성
# =========================================================

# ---------- 유틸 1) 상관 기반 팩터 축소 ----------
def reduce_factors_by_corr(df, factors, method="spearman", threshold=0.85, prefer=None):
    """
    factors: 후보 팩터 리스트
    threshold: |corr|가 이 값보다 크면 하나 제거
    prefer: 남기고 싶은 팩터 우선순위 dict (높을수록 남김)
            예) {"d_margin_ratio_20": 10, "inst_flow_roll20": 5, ...}
    """
    X = df[factors].copy()
    corr = X.corr(method=method).abs()

    to_drop = set()
    cols = list(corr.columns)

    # prefer 기본값(없으면 모두 0)
    if prefer is None:
        prefer = {c: 0 for c in cols}
    else:
        for c in cols:
            prefer.setdefault(c, 0)

    # 상관 높은 pair를 찾아 하나씩 제거
    # (단순 greedy 방식: i<j pair 순회)
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            a, b = cols[i], cols[j]
            if a in to_drop or b in to_drop:
                continue
            if corr.loc[a, b] >= threshold:
                # prefer가 낮은 쪽을 제거 (동점이면 IC/분산 등 추가 규칙 넣어도 됨)
                if prefer[a] > prefer[b]:
                    to_drop.add(b)
                elif prefer[b] > prefer[a]:
                    to_drop.add(a)
                else:
                    # prefer 동점이면 roll20을 선호(예시), 아니면 b 제거
                    # 원하는 규칙으로 바꿔도 됨
                    if "roll20" in a and "roll20" not in b:
                        to_drop.add(b)
                    elif "roll20" in b and "roll20" not in a:
                        to_drop.add(a)
                    else:
                        to_drop.add(b)

    kept = [c for c in factors if c not in to_drop]
    return kept, corr


# ---------- 유틸 2) Ridge 조합 모델 학습/스코어 ----------
def fit_ridge_score(df, factors, target="fwd_ret_20", alpha=5.0, min_rows=10000):
    """
    전체 기간 한번에 학습한 ridge로 score 생성(빠름).
    OOS 고려는 4.3에서 하거나, 아래 워크포워드 버전을 사용.
    """
    data = df[["date","ticker", target] + factors].dropna().copy()
    if len(data) < min_rows:
        print(f"[WARN] train rows={len(data)} < min_rows={min_rows}. 그래도 진행합니다.")

    X = data[factors].values
    y = data[target].values

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])
    model.fit(X, y)

    coef = pd.Series(model.named_steps["ridge"].coef_, index=factors).sort_values(key=lambda x: x.abs(), ascending=False)

    # score 생성
    out = df.copy()
    out["score_ridge"] = np.nan
    m = out[factors].notna().all(axis=1)
    out.loc[m, "score_ridge"] = model.predict(out.loc[m, factors].values)

    return out, model, coef


# ---------- 유틸 3) 워크포워드 Ridge 스코어 ----------
def fit_ridge_score_walkforward(df, factors, target="fwd_ret_20",
                                train_years=1, step_months=1, alpha=5.0,
                                min_train_rows=20000):
    """
    누수 방지: 매 스텝마다 과거 구간만으로 학습 -> 다음 구간 score 예측
    - train_years: 학습에 사용할 과거 기간(년)
    - step_months: 예측을 생성할 앞으로의 기간(개월)
    """
    out = df.sort_values(["date","ticker"]).copy()
    out["score_ridge_wf"] = np.nan

    # month 컬럼을 미리 만들어서 안정적으로 필터
    out["month"] = out["date"].dt.to_period("M")

    months = pd.Index(out["month"].dropna().unique()).sort_values()

    # 월별 루프
    for idx in range(len(months)):
        # --- 경계: 다음 인덱스가 범위를 벗어나거나 정확히 끝이면 중단
        if idx + step_months >= len(months):
            break

        test_start_m = months[idx]
        test_end_m   = months[idx + step_months]   # 이 월 "미만" 까지가 테스트 구간

        train_start_m = test_start_m - train_years*12
        train_end_m   = test_start_m - 1           # 직전 월까지

        train_mask = (out["month"] >= train_start_m) & (out["month"] <= train_end_m)
        test_mask  = (out["month"] >= test_start_m) & (out["month"] <  test_end_m)

        train = out.loc[train_mask, ["date","ticker", target] + factors].dropna()
        test  = out.loc[test_mask,  ["date","ticker"] + factors].dropna()

        if len(train) < min_train_rows or len(test) == 0:
            continue

        Xtr = train[factors].values
        ytr = train[target].values
        Xte = test[factors].values

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha))
        ])
        model.fit(Xtr, ytr)
        preds = model.predict(Xte)

        out.loc[test.index, "score_ridge_wf"] = preds

    # 정리: month 컬럼 제거하고 반환
    out = out.drop(columns=["month"])
    return out


# =========================================================
# 1) 4.1 결과 기반으로 후보 팩터 리스트 세팅
#    (너가 붙여준 결과 기준 추천 기본 세트)
# =========================================================
candidate_factors = [
    "d_margin_ratio_20",
    "d_short_ratio_5",
    "inst_flow_roll5",
    "inst_flow_roll20",
    "foreign_flow_roll20",
    "d_borrow_sh_20",
    # 필요하면 여기도 추가 가능:
    # "ret_20", "ret_5", "d_borrow_amt_20", ...
]

# 남기고 싶은 우선순위(원하는 대로 조정 가능)
prefer = {
    "d_margin_ratio_20": 10,
    "d_short_ratio_5": 9,
    "inst_flow_roll20": 6,
    "inst_flow_roll5": 5,
    "foreign_flow_roll20": 7,
    "d_borrow_sh_20": 7,
}


# =========================================================
# 2) 공선성 제거
# =========================================================
use_factors, corr_mat = reduce_factors_by_corr(
    df, candidate_factors, method="spearman", threshold=0.85, prefer=prefer
)

print("=== Candidate factors ===")
print(candidate_factors)
print("\n=== Kept factors (after corr filter) ===")
print(use_factors)

print("\n=== Spearman |corr| matrix (candidate) ===")
print(corr_mat)


# =========================================================
# 3) Ridge로 조합 스코어 생성 (빠른 버전)
# =========================================================
df2, ridge_model, coef_s = fit_ridge_score(
    df, use_factors, target="fwd_ret_20", alpha=5.0
)

print("\n=== Ridge coefficients (abs desc) ===")
print(coef_s)

# df2["score_ridge"] 가 생성됨


# =========================================================
# 4) (강추) 워크포워드로 누수 방지한 스코어 생성
# =========================================================
df2 = fit_ridge_score_walkforward(
    df2, use_factors, target="fwd_ret_20",
    train_years=1, step_months=1, alpha=5.0,
    min_train_rows=20000
)

print("\nDone. df2 columns added: score_ridge, score_ridge_wf")


=== Candidate factors ===
['d_margin_ratio_20', 'd_short_ratio_5', 'inst_flow_roll5', 'inst_flow_roll20', 'foreign_flow_roll20', 'd_borrow_sh_20']

=== Kept factors (after corr filter) ===
['d_margin_ratio_20', 'd_short_ratio_5', 'inst_flow_roll5', 'inst_flow_roll20', 'foreign_flow_roll20', 'd_borrow_sh_20']

=== Spearman |corr| matrix (candidate) ===
                     d_margin_ratio_20  d_short_ratio_5  inst_flow_roll5  \
d_margin_ratio_20             1.000000         0.072366         0.166091   
d_short_ratio_5               0.072366         1.000000         0.033581   
inst_flow_roll5               0.166091         0.033581         1.000000   
inst_flow_roll20              0.185418         0.031482         0.664192   
foreign_flow_roll20           0.192048         0.159502         0.200388   
d_borrow_sh_20                0.065528         0.147640         0.012485   

                     inst_flow_roll20  foreign_flow_roll20  d_borrow_sh_20  
d_margin_ratio_20            0.18541

### 2단계 결과해석

**(1) 공선성**

- inst_flow_roll5 <-> inst_flow_roll20 스피어만 상관 0.664  
  -> 높긴 하지만 0.85 기준 미만이라 둘 다 유지 가능  
  다만 전략에선 둘이 동시에 들어가면 신호가 중복될 수 있으니 3단계에서 “둘 다 vs 하나만” 비교하면 좋음

**(2) Ridge 계수 방향이 1단계 IC와 다르게 나온 이유**

1단계에서는 단일 팩터의 단면 IC였고,  
2단계 Ridge는 여러 팩터를 동시에 넣고 표준화한 뒤 “겹치는 설명력”을 서로 나눠 갖는 구조라서

- 어떤 팩터는 단독일 때는 +, 같이 넣으면 -로 뒤집히는 일이 흔함
- 특히 inst_flow_roll5(+)가 가장 크고, inst_flow_roll20(-)가 음으로 잡힌 건  
  “단기 매수 강도는 플러스인데, 동일 조건에서 장기 강도는 오히려 mean-reversion 성분”으로 잡혔을 가능성이 있음
- foreign_flow_roll20은 4.1에서 -였는데, Ridge에선 +로 아주 작게 잡힘  
  → “다른 변수 통제 후”에는 방향이 바뀐 것  
  즉, 외국인 단독 신호는 약하고 다른 신호들과 섞일 때 의미가 달라질 수 있음

**3단계 전략 후보**

- Ridge 스코어 기반 TopN 롱온리
- inst_flow_roll5 단일 팩터 TopN (가장 강한 계수)
- d_borrow_sh_20 회피(하위 회피) 또는 역방향 TopN

## 3단계: 워크포워드 스코어로 TopN 롱온리 백테스트



In [5]:
import numpy as np
import pandas as pd

def get_rebal_dates(dates: pd.Series, freq="W-FRI"):
    """freq='W-FRI' (주간), 'M' (월말) 등"""
    s = pd.Series(pd.to_datetime(dates.unique())).sort_values()
    tmp = pd.DataFrame({"date": s})
    # 각 리밸 주기 버킷에서 마지막 거래일 선택
    rebal = tmp.groupby(pd.Grouper(key="date", freq=freq))["date"].max().dropna()
    return pd.DatetimeIndex(rebal.values)

def backtest_topN_long_only(
    df,
    score_col="score_ridge_wf",
    price_col="종가",
    freq="W-FRI",
    topN=20,
    cost_per_turnover=0.002,   # turnover 100%일 때 비용 0.2% (왕복 0.1%씩 가정)
    min_names=50
):
    """
    롱온리 TopN 동일비중.
    - score_col: 클수록 매수 선호
    - cost_per_turnover: 턴오버 1.0일 때 적용할 비용(예: 0.002 = 0.2%)
    """
    data = df.sort_values(["date","ticker"]).copy()
    data["date"] = pd.to_datetime(data["date"])

    # 일별 수익률
    data["ret_1"] = data.groupby("ticker")[price_col].pct_change().fillna(0)

    # 리밸런싱 날짜
    rebal_dates = get_rebal_dates(data["date"], freq=freq)
    rebal_dates = [d for d in rebal_dates if d in set(data["date"].unique())]

    # 포트폴리오 일별 수익률 기록
    port = []
    prev_w = {}  # 이전 비중 dict

    # 날짜 인덱스를 빠르게 쓰기 위해 그룹
    by_date = {d: g for d, g in data.groupby("date")}

    # 리밸런싱 구간 루프
    for i in range(len(rebal_dates)-1):
        d0 = rebal_dates[i]
        d1 = rebal_dates[i+1]

        snap = by_date.get(d0)
        if snap is None:
            continue

        snap = snap.dropna(subset=[score_col])
        if len(snap) < max(topN, min_names):
            continue

        picks = snap.nlargest(topN, score_col)["ticker"].tolist()
        w = {t: 1.0/topN for t in picks}

        # 턴오버 계산(0.5*|Δw| 합)
        all_t = set(prev_w.keys()) | set(w.keys())
        turnover = 0.5 * sum(abs(w.get(t,0.0) - prev_w.get(t,0.0)) for t in all_t)
        cost = turnover * cost_per_turnover

        # d0 다음날부터 d1까지(리밸 날짜 포함/미포함 규칙 선택)
        # 여기선 "d0 당일 종가로 리밸 -> d0+1부터 수익 반영"을 근사하기 위해
        # d0 다음 거래일부터 d1까지 수익을 누적
        dates_in_period = [d for d in by_date.keys() if (d > d0 and d <= d1)]
        dates_in_period = sorted(dates_in_period)

        for d in dates_in_period:
            g = by_date[d]
            g = g[g["ticker"].isin(picks)]
            if len(g) == 0:
                continue
            # 동일비중 평균 수익
            r = g["ret_1"].mean()
            port.append([d, r])

        # 비용은 리밸런싱 직후 하루에 차감(가장 단순한 방식)
        if len(port) > 0:
            port[-1][1] -= cost

        prev_w = w

    port_df = pd.DataFrame(port, columns=["date","port_ret"]).drop_duplicates("date").set_index("date").sort_index()
    port_df["cum"] = (1 + port_df["port_ret"]).cumprod()
    return port_df


**3개 전략 비교**

In [6]:
# 1) Ridge 워크포워드 스코어
bt_ridge = backtest_topN_long_only(df2, score_col="score_ridge_wf", freq="W-FRI", topN=20, cost_per_turnover=0.002)

# 2) 단일 팩터: inst_flow_roll5 (클수록 좋다 가정)
bt_inst5 = backtest_topN_long_only(df2, score_col="inst_flow_roll5", freq="W-FRI", topN=20, cost_per_turnover=0.002)

# 3) 대차 변화는 Ridge에서 음(-)이었으니 "작을수록 좋다"면 score를 -로 뒤집어서 사용
df2["neg_d_borrow_sh_20"] = -df2["d_borrow_sh_20"]
bt_borrow = backtest_topN_long_only(df2, score_col="neg_d_borrow_sh_20", freq="W-FRI", topN=20, cost_per_turnover=0.002)

print("ridge final cum:", bt_ridge["cum"].iloc[-1])
print("inst5 final cum:", bt_inst5["cum"].iloc[-1])
print("borrow final cum:", bt_borrow["cum"].iloc[-1])


ridge final cum: 1.6057452728179142
inst5 final cum: 1.1839517562845816
borrow final cum: 1.3277637612901239


### 3단계 결과해석

| 전략 | 최종 누적 | 해석 |
|--|--|--|
| Ridge 조합 (score_ridge_wf) | 1.61x | ✅ 확실한 알파 |
| inst_flow_roll5 단일 | 1.18x | ⚠️ 약한 추세 |
| d_borrow_sh_20 단일 | 1.33x | ◯ 보조 신호 |

### 추가 검증


**3.1 리밸런싱 민감도**

- 주간만 좋고 월간에서 깨지면 → 과최적화 의심
- 주/월 둘 다 안정적이면 → 실전 OK

In [8]:
def perf_stats(port_df, ann_factor=252):
    r = port_df["port_ret"].dropna()
    if len(r)==0:
        return {}
    cum = (1+r).cumprod()
    dd = cum / cum.cummax() - 1
    cagr = cum.iloc[-1]**(ann_factor/len(r)) - 1
    vol  = r.std(ddof=1) * np.sqrt(ann_factor)
    sharpe = (r.mean() * ann_factor) / (r.std(ddof=1) * np.sqrt(ann_factor) + 1e-12)
    mdd = dd.min()
    return {"CAGR": cagr, "VOL": vol, "Sharpe": sharpe, "MDD": mdd, "FinalCum": cum.iloc[-1], "N": len(r)}

print("Ridge:", perf_stats(bt_ridge))
print("Inst5:", perf_stats(bt_inst5))
print("Borrow:", perf_stats(bt_borrow))

Ridge: {'CAGR': 0.21570721917218982, 'VOL': 0.2127686955513103, 'Sharpe': 1.0249948401173075, 'MDD': -0.2592532884824934, 'FinalCum': 1.6057452728179142, 'N': 611}
Inst5: {'CAGR': 0.06036352200120487, 'VOL': 0.16617116761159614, 'Sharpe': 0.43590014512380415, 'MDD': -0.1873967304750649, 'FinalCum': 1.1839517562845816, 'N': 726}
Borrow: {'CAGR': 0.10617200114255154, 'VOL': 0.20115530387205394, 'Sharpe': 0.6026217903379016, 'MDD': -0.23370698275671642, 'FinalCum': 1.3277637612901239, 'N': 708}


In [10]:
bt_ridge_M = backtest_topN_long_only(
    df2, score_col="score_ridge_wf",
    freq="ME", topN=20, cost_per_turnover=0.002
)
print("Ridge Monthly:", perf_stats(bt_ridge_M))

Ridge Monthly: {'CAGR': 0.20624258235758086, 'VOL': 0.21561582645970329, 'Sharpe': 0.9779111387699233, 'MDD': -0.1643596304837387, 'FinalCum': 1.5720894910893626, 'N': 608}


**리밸런싱 민감도 결과**

- CAGR: 20.6%
- VOL: 21.6%
- Sharpe: 0.98
- MDD: -16.4%
- FinalCum: 1.57
- 기간 N : 608 거래일

**FinalCum = 1.57**  
- 초기자산 대비 +57%
- 월간 리밸런싱에서도 주간과 거의 동일한 누적 성과
  - (주간: 1.60 / 월간: 1.57)  
👉 리밸런싱 주기에 민감하지 않다  
→ 과최적화 가능성 ↓

**CAGR ≈ 20.6%**  
👉 명확한 초과수익

**MDD ≈ -16.4%**  
👉 리스크 대비 성과 구조가 매우 건강

**Sharpe ≈ 0.98**  
👉 좋음과 아쉬움의 경계

**3.2 TopN 민감도 (10 / 20 / 30)**

- Top10만 유독 좋으면 → 특정 종목 의존
- Top20~30이 가장 안정적이면 → 진짜 팩터

In [11]:
for n in [10, 20, 30]:
    bt = backtest_topN_long_only(
        df2, score_col="score_ridge_wf",
        freq="W-FRI", topN=n, cost_per_turnover=0.002
    )
    print(f"Top{n}:", perf_stats(bt))

Top10: {'CAGR': 0.17449667575612593, 'VOL': 0.2571808899103045, 'Sharpe': 0.7545929988250722, 'MDD': -0.4810708910343614, 'FinalCum': 1.476940080772091, 'N': 611}
Top20: {'CAGR': 0.21570721917218982, 'VOL': 0.2127686955513103, 'Sharpe': 1.0249948401173075, 'MDD': -0.2592532884824934, 'FinalCum': 1.6057452728179142, 'N': 611}
Top30: {'CAGR': 0.15009638273465198, 'VOL': 0.19858622211252294, 'Sharpe': 0.8040437537803887, 'MDD': -0.2350154769124453, 'FinalCum': 1.403642227273984, 'N': 611}


**TopN 민감도 결과**

| Top N	| CAGR | Sharpe | MDD | FinalCum | 해석 |
|--|--|--|--|--|--|
| Top10 | 17.4% | 0.75 | -48% | 1.48 | ❌ 집중 리스크 과도 |
| Top20 | 21.6% | 1.02 | -25.9% | 1.61 | ✅ 최적 |
| Top30 | 15.0% | 0.80 | -23.5% | 1.40 | ⚠️ 희석 효과 |

👉 Top20이 알파 + 분산의 균형점

**3.3 비용 민감도 (현실성 체크)**

- 0.3%에서 무너지면 → 실전성 낮음
- 0.2~0.3%에서도 유지되면 → 충분히 현실적

In [12]:
for c in [0.001, 0.002, 0.003]:
    bt = backtest_topN_long_only(
        df2, score_col="score_ridge_wf",
        freq="W-FRI", topN=20, cost_per_turnover=c
    )
    print(f"Cost {c}:", perf_stats(bt))


Cost 0.001: {'CAGR': 0.24823248931032071, 'VOL': 0.21254494356738926, 'Sharpe': 1.1501674001610973, 'MDD': -0.24348741626759152, 'FinalCum': 1.7118997642696017, 'N': 611}
Cost 0.002: {'CAGR': 0.21570721917218982, 'VOL': 0.2127686955513103, 'Sharpe': 1.0249948401173075, 'MDD': -0.2592532884824934, 'FinalCum': 1.6057452728179142, 'N': 611}
Cost 0.003: {'CAGR': 0.1840131080539147, 'VOL': 0.21304399134726024, 'Sharpe': 0.8998675052251872, 'MDD': -0.274698767407762, 'FinalCum': 1.5061229562535419, 'N': 611}


**비용 민감도 결과**

| 비용 | CAGR | Sharpe | MDD | FinalCum |
|--|--|--|--|--|
| 0.1% | 24.8% | 1.15 | -24.3% | 1.71 |
| 0.2% | 21.6% | 1.02 | -25.9% | 1.61 |
| 0.3% | 18.4% | 0.90 | -27.5% | 1.51 |

**해석**  
- 비용 증가 → 성과가 ‘부드럽게’ 감소
- Sharpe, CAGR 모두 선형적으로 감소
- 어느 지점에서도 “전략 붕괴” 없음

👉 좋은 신호

## 결론

여러 수급·포지션 지표를 결합해 만든 score_ridge_wf를  
하나의 종합 팩터(Composite Factor)로 사용해  
Top20 롱온리 전략을 운용하라

**score_ridge_wf** 정의
- 여러 원천 데이터(기관·외국인·공매도·대차·신용)를 표준화하고
- 워크포워드 방식으로 결합해
- 각 종목에 대해 매 시점 하나의 점수를 만듦

**사용한 원천 팩터(입력 변수)**
- d_margin_ratio_20 : 최근 20일 신용비율 변화
- d_short_ratio_5 : 최근 5일 공매도 비중 변화
- inst_flow_roll5 : 기관 순매수 강도 (5일 평균, 거래대금 정규화)
- inst_flow_roll20 : 기관 순매수 강도 (20일 평균)
- foreign_flow_roll20 : 외국인 순매수 강도 (20일 평균)
- d_borrow_sh_20 : 대차잔고 주식수 변화 (20일)

**가공 방법**
- 1단계: 표준화(z-score)
- 2단계: Ridge 회귀 학습 (가중치 추정)
  - 타깃변수: 향후 20영업일 수익률 (fwd_ret_20)
- 3단계: 워크포워드 적용
  - 각 월다마 과거 1년 데이터만 사용해서 Ridge 학습
  - 해당 계수 고정
  - 다음 1개월 구간의 종목 점수 계산

**전략 적용 방법**
- 매 리밸런싱 시점 t에서
- score_ridge_wf_{i,t} 계산
- 점수 기준 상위 20개 종목 선택
- 동일비중 매수
- 다음 리밸런싱까지 보유